# Task 2 — Paper Reproduction: Retrieval-Augmented Generation
Dataset: TriviaQA (`rc.nocontext`) — Project 10

Paper: Lewis, Perez, Piktus, et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, arXiv:2005.11401 (NeurIPS 2020).

## 0. Overview

Run on Colab (GPU). Upload `triviaqa_sample_300.jsonl` and `wiki_paragraph_index.jsonl` when
prompted. Run locally (no GPU) to reuse an already-downloaded `task2_generation_results.jsonl`.

## 1. Setup

In [1]:
# Importing the libraries we need for data handling and evaluation
import sys
import json
import re
import string
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

# Fixing the seed everywhere so retrieval/generation stays reproducible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Detecting whether we are running on Colab (needed for the GPU generator cells)
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running on Colab:" if IN_COLAB else "Running locally:", IN_COLAB)

Running locally: False


In [2]:
# Installing the extra packages we need on top of Colab's preinstalled torch/transformers
if IN_COLAB:
    # torch/transformers come preinstalled on Colab GPU runtimes; add the rest.
    get_ipython().system('pip install -q sentence-transformers faiss-cpu accelerate "bitsandbytes>=0.46.1"')
    print("Installed Colab-only dependencies. If this is the FIRST time bitsandbytes was "
          "installed in this session, restart the runtime now (Runtime > Restart session) "
          "and re-run from this cell -- a fresh import is required for its compiled extension "
          "to load correctly.")
else:
    print("Local run: assuming `pip install -r ../requirements.txt` has already been run.")

Local run: assuming `pip install -r ../requirements.txt` has already been run.


In [3]:
# Setting up the config constants used throughout this notebook
K_LIST = [1, 3, 5, 10]
TOP_K = 5            # spec: dense retriever / generator top-k=5
TOP_K_RETRIEVE = 10  # retrieve enough to also compute recall@10
MAX_PASSAGE_TOKENS = 200   # spec: each passage truncated to 200 tokens before the generator
MAX_NEW_TOKENS = 64        # spec
TEMPERATURE = 0.0          # spec -> greedy decoding
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Setting up the filenames shared with Task 1 (same sample, same index)
SAMPLE_FILENAME = "triviaqa_sample_300.jsonl"
INDEX_FILENAME = "wiki_paragraph_index.jsonl"
GENERATION_RESULTS_FILENAME = "task2_generation_results.jsonl"

# On Colab there's no shared data/ folder, so prompt for the two Task 1 files directly
if IN_COLAB:
    from google.colab import files
    print(f"Upload {SAMPLE_FILENAME} and {INDEX_FILENAME} (from this repo's data/ folder):")
    uploaded = files.upload()
    assert SAMPLE_FILENAME in uploaded and INDEX_FILENAME in uploaded, (
        "Both files are required -- re-run this cell and upload them."
    )
    SAMPLE_PATH = Path(SAMPLE_FILENAME)
    INDEX_PATH = Path(INDEX_FILENAME)
    RESULTS_DIR = Path(".")  # Colab session working directory
else:
    # Local run: reuse the data/ folder Task 1 already populated
    DATA_DIR = Path("../../23cs30011_project/data")
    RESULTS_DIR = Path("../../23cs30011_project/results")
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    SAMPLE_PATH = DATA_DIR / SAMPLE_FILENAME
    INDEX_PATH = DATA_DIR / INDEX_FILENAME

GENERATION_RESULTS_PATH = RESULTS_DIR / GENERATION_RESULTS_FILENAME

# Loading the fixed 300-question sample and the pooled Wikipedia paragraph index from Task 1
samples = [json.loads(l) for l in open(SAMPLE_PATH)]
passages = [json.loads(l) for l in open(INDEX_PATH)]
print(f"Loaded {len(samples)} questions (same fixed sample as Task 1).")
print(f"Loaded {len(passages)} paragraphs (same pooled Wikipedia index as Task 1).")

Loaded 300 questions (same fixed sample as Task 1).
Loaded 7660 paragraphs (same pooled Wikipedia index as Task 1).


## 2. Dense retrieval: `all-MiniLM-L6-v2` + FAISS flat, cosine similarity

In [4]:
from sentence_transformers import SentenceTransformer
import faiss

# Loading the sentence embedder used for dense retrieval
embedder = SentenceTransformer(EMBED_MODEL_NAME)
print("Embedding dim:", embedder.get_sentence_embedding_dimension())

# Embedding every passage in the pool once, up front, so FAISS can index them
passage_texts = [p["text"] for p in passages]
passage_embeddings = embedder.encode(
    passage_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

# Building a flat (exact) FAISS index over the normalized embeddings -- inner product
# on unit vectors is just cosine similarity
dim = passage_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(passage_embeddings)
print("FAISS flat index built:", faiss_index.ntotal, "vectors, dim", dim)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dim: 384


/tmp/ipykernel_169397/658966525.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dim:", embedder.get_sentence_embedding_dimension())


Batches:   0%|          | 0/120 [00:00<?, ?it/s]

FAISS flat index built: 7660 vectors, dim 384


## 3. Retrieve top-k for every sampled question, evaluate recall@k

In [5]:
# Embedding every question the same way the passages were embedded
question_texts = [s["question"] for s in samples]
question_embeddings = embedder.encode(
    question_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

# Searching the FAISS index for each question's top-k nearest passages
scores, indices = faiss_index.search(question_embeddings, TOP_K_RETRIEVE)
dense_retrieval_results = [
    list(zip(indices[i].tolist(), scores[i].tolist())) for i in range(len(samples))
]

# Quick sanity check on the first question's retrieved passages
print("Example dense retrieval for:", samples[0]["question"])
for pid, sc in dense_retrieval_results[0][:3]:
    print(f"  cos_sim={sc:.3f} doc={passages[pid]['doc_title']!r} text={passages[pid]['text'][:120]!r}")

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Example dense retrieval for: Which actress was voted Miss Greenwich Village in 1942?
  cos_sim=0.387 doc='Lauren Bacall' text='In 1941, Bacall studied for a year at the American Academy of Dramatic Arts in New York, where she met classmate and fut'
  cos_sim=0.376 doc='Lauren Bacall' text='She returned to Broadway in the musical Woman of the Year (1981) with book by Peter Stone and music and lyrics by Kander'
  cos_sim=0.374 doc='Una Stubbs' text='Una Stubbs (1 May 1937 – 12 August 2021) was an English actress, television personality, and dancer who appeared on Brit'


In [6]:
def normalize_answer(s):
    """SQuAD-style normalization: lowercase, remove punctuation, remove articles, fix whitespace."""

    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s


def recall_at_k(hits, golds, k, passages_ref):
    """Utility function to check whether any gold answer string appears
    (as a normalized substring) in any of the top-k retrieved passages."""

    norm_golds = [normalize_answer(g) for g in golds]
    for pid, _ in hits[:k]:
        norm_text = normalize_answer(passages_ref[pid]["text"])
        if any(g and g in norm_text for g in norm_golds):
            return 1.0
    return 0.0


# Iterating through every question and computing recall@k against the dense retriever's hits
recall_records = []
for s, hits in zip(samples, dense_retrieval_results):
    rec = {"question_id": s["question_id"]}
    for k in K_LIST:
        rec[f"recall@{k}"] = recall_at_k(hits, s["gold_answers"], k, passages)
    recall_records.append(rec)

recall_df = pd.DataFrame(recall_records)
dense_recall_metrics = {f"recall@{k}": recall_df[f"recall@{k}"].mean() for k in K_LIST}

print("=== Task 2 dense retrieval (all-MiniLM-L6-v2, FAISS flat, cosine) recall@k ===")
for k, v in dense_recall_metrics.items():
    print(f"{k:>10}: {v:.4f}")

=== Task 2 dense retrieval (all-MiniLM-L6-v2, FAISS flat, cosine) recall@k ===
  recall@1: 0.5133
  recall@3: 0.6200
  recall@5: 0.6700
 recall@10: 0.7300


## 4. Generation: top-5 passages (200 tokens each) -> `Qwen2.5-7B-Instruct` (4-bit), greedy decode

In [7]:
if IN_COLAB:
    import time
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    torch.manual_seed(SEED)

    # Loading the generator in 4-bit (bitsandbytes NF4) to fit comfortably on a Colab T4 GPU
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    # Loading the tokenizer and the quantized generator model
    print("Loading tokenizer/model:", GEN_MODEL_NAME)
    gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
    gen_model = AutoModelForCausalLM.from_pretrained(
        GEN_MODEL_NAME, quantization_config=quant_config, device_map="auto",
    )
    gen_model.eval()

    def truncate_to_tokens(text, max_tokens):
        """Utility function that truncates text down to at most max_tokens generator tokens."""
        ids = gen_tokenizer.encode(text, add_special_tokens=False)
        if len(ids) <= max_tokens:
            return text
        return gen_tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)

    def build_prompt(question, top_passages):
        """Utility function that builds the zero-shot chat prompt: the top-k passages
        (each truncated to MAX_PASSAGE_TOKENS) followed by the question."""

        passage_block = "\n\n".join(
            f"Passage {i+1} ({passages[pid]['doc_title']}): "
            f"{truncate_to_tokens(passages[pid]['text'], MAX_PASSAGE_TOKENS)}"
            for i, (pid, _) in enumerate(top_passages[:TOP_K])
        )
        user_msg = (
            "Answer the question using ONLY the information in the passages below. "
            "Respond with just the short answer, no explanation, no full sentence.\n\n"
            f"{passage_block}\n\nQuestion: {question}\nAnswer:"
        )
        messages = [
            {"role": "system", "content": "You are a concise, factual question-answering assistant."},
            {"role": "user", "content": user_msg},
        ]
        return gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def generate_answer(question, top_passages):
        """Utility function that builds the prompt and greedily decodes the generator's answer."""

        prompt = build_prompt(question, top_passages)
        inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
        with torch.no_grad():
            out = gen_model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=gen_tokenizer.eos_token_id,
            )
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        return gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Iterating through every sampled question, generating an answer from its top-k passages
    generations = []
    t0 = time.time()
    for i, (s, hits) in enumerate(zip(samples, dense_retrieval_results)):
        pred = generate_answer(s["question"], hits)
        generations.append({
            "question_id": s["question_id"], "question": s["question"],
            "gold_answers": s["gold_answers"], "prediction": pred,
        })
        if (i + 1) % 10 == 0:
            elapsed = time.time() - t0
            print(f"{i+1}/{len(samples)} done, {elapsed:.0f}s elapsed, ~{elapsed/(i+1):.1f}s/question")

    # Persisting the raw generations so Tasks 2/3 scoring can be re-run without regenerating
    with open(GENERATION_RESULTS_PATH, "w") as f:
        for r in generations:
            f.write(json.dumps(r) + "\n")
    print("Saved", GENERATION_RESULTS_PATH)

    print("\nDownload this file and place it in this repo's results/ folder:")
    files.download(str(GENERATION_RESULTS_PATH))

else:
    # Local run: no GPU here, so just reuse a previous Colab run's generations
    assert GENERATION_RESULTS_PATH.exists(), (
        f"{GENERATION_RESULTS_PATH} not found. Run this notebook on Colab (GPU) first to "
        "generate it, or place a previous run's output there."
    )
    generations = [json.loads(l) for l in open(GENERATION_RESULTS_PATH)]
    print(f"Local run: loaded {len(generations)} pre-generated predictions from "
          f"{GENERATION_RESULTS_PATH} (generation itself requires the Colab GPU run).")

# Indexing the generations by question_id and checking nothing got dropped/added
gen_by_qid = {g["question_id"]: g for g in generations}
assert set(gen_by_qid) == {s["question_id"] for s in samples}, "generation set doesn't match the sample set"
print("Question IDs match the fixed sample exactly.")

Local run: loaded 300 pre-generated predictions from ../../results/task2_generation_results.jsonl (generation itself requires the Colab GPU run).
Question IDs match the fixed sample exactly.


## 5. Score generations: EM, F1, recall@k

In [8]:
def compute_em(pred, golds):
    """Utility function to return 1.0 if the prediction exactly matches any
    gold answer (after normalization), else 0.0."""

    norm_pred = normalize_answer(pred)
    return float(any(norm_pred == normalize_answer(g) for g in golds))


def f1_score(pred, gold):
    """Utility function that computes token-level F1 between one prediction
    and one gold answer string."""

    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_f1(pred, golds):
    """Utility function that takes the best F1 across all acceptable gold answers."""
    return max(f1_score(pred, g) for g in golds) if golds else 0.0


# Iterating through every question and building one results record: gold answers,
# the LLM's prediction, EM/F1, and recall@k for every k we track
records = []
for s, hits in zip(samples, dense_retrieval_results):
    g = gen_by_qid[s["question_id"]]
    pred = g["prediction"]
    golds = s["gold_answers"]
    rec = {
        "question_id": s["question_id"],
        "question": s["question"],
        "gold_answers": golds,
        "prediction": pred,
        "em": compute_em(pred, golds),
        "f1": compute_f1(pred, golds),
        "top1_doc": passages[hits[0][0]]["doc_title"],
    }
    for k in K_LIST:
        rec[f"recall@{k}"] = recall_at_k(hits, golds, k, passages)
    records.append(rec)

task2_results_df = pd.DataFrame(records)
task2_results_df.to_csv(RESULTS_DIR / "task2_per_question_results.csv", index=False)
task2_results_df.head(5)

,question_id,question,gold_answers,prediction,em,f1,top1_doc,recall@1,recall@3,recall@5,recall@10
0,tc_106,Which actress was voted Miss Greenwich Village...,"[Lauren Bacall, Bacall, Lauren Becal, Lauren B...",Lauren Bacall,1.0,1.0,Lauren Bacall,1.0,1.0,1.0,1.0
1,tc_1331,Who had a 60s No 1 hit with This Diamond Ring?,"[Gary Lewis and the Playboys, Dave Gonzalez (G...",Gary Lewis & the Playboys,1.0,1.0,Gary Lewis & the Playboys,1.0,1.0,1.0,1.0
2,tc_2020,What percentage of the earth's surface is cove...,"[8%, eight %, 8 %, 8, eight]",10.9%,0.0,0.0,Iceland,1.0,1.0,1.0,1.0
3,tc_2167,Which US city was named after a British Prime ...,"[Pittsburgh, Smoky City, Pittsburgh (Pa.), Pit...",Pittsburgh,1.0,1.0,Pittsburgh,1.0,1.0,1.0,1.0
4,tc_2900,"Which teeth are the third molars, and are alwa...","[Wisdom teeth, Wisdom-tooth, Third maxillary m...",Wisdom teeth,1.0,1.0,Wisdom tooth,1.0,1.0,1.0,1.0


In [9]:
# Averaging EM/F1/recall@k over all 300 questions and saving the summary metrics
task2_metrics = {
    "n_questions": len(task2_results_df),
    "EM": task2_results_df["em"].mean(),
    "F1": task2_results_df["f1"].mean(),
}
for k in K_LIST:
    task2_metrics[f"recall@{k}"] = task2_results_df[f"recall@{k}"].mean()

with open(RESULTS_DIR / "task2_metrics.json", "w") as f:
    json.dump(task2_metrics, f, indent=2)

print("=== Task 2 (TriviaQA rc.nocontext) — dense retrieval + LLM generation ===")
for key, val in task2_metrics.items():
    print(f"{key:>12}: {val:.4f}" if isinstance(val, float) else f"{key:>12}: {val}")

# side-by-side with Task 1, if its metrics file is reachable (local run only)
task1_metrics_path = Path("../../results/task1_metrics.json")
if task1_metrics_path.exists():
    with open(task1_metrics_path) as f:
        task1_metrics = json.load(f)
    comparison_df = pd.DataFrame({"Task 1 (BM25 + heuristic)": task1_metrics, "Task 2 (dense + LLM)": task2_metrics})
    print("\n=== Task 1 vs Task 2 ===")
    print(comparison_df)

if IN_COLAB:
    print("\nDownload these files and place them in this repo's results/ folder:")
    files.download(str(RESULTS_DIR / "task2_per_question_results.csv"))
    files.download(str(RESULTS_DIR / "task2_metrics.json"))

=== Task 2 (TriviaQA rc.nocontext) — dense retrieval + LLM generation ===
 n_questions: 300
          EM: 0.6600
          F1: 0.6979
    recall@1: 0.5133
    recall@3: 0.6200
    recall@5: 0.6700
   recall@10: 0.7300

=== Task 1 vs Task 2 ===
             Task 1 (BM25 + heuristic)  Task 2 (dense + LLM)
n_questions                 300.000000            300.000000
EM                            0.020000              0.660000
F1                            0.092664              0.697888
recall@1                      0.533333              0.513333
recall@3                      0.660000              0.620000
recall@5                      0.716667              0.670000
recall@10                     0.743333              0.730000


## 6. Qualitative analysis: correct vs. incorrect examples

In [10]:
def show_examples(df, n=5, correct=True, seed=SEED):
    """Utility function to print n random correct (EM=1) or incorrect (EM=0)
    examples from the results dataframe, for qualitative inspection."""

    subset = df[df["em"] == (1.0 if correct else 0.0)]
    subset = subset.sample(min(n, len(subset)), random_state=seed)
    # Iterating through the sampled rows and printing question/gold/pred side by side
    for _, row in subset.iterrows():
        print(f"Q: {row['question']}")
        print(f"   Gold: {row['gold_answers']}")
        print(f"   Pred: {row['prediction']!r}")
        print(f"   Top-1 doc: {row['top1_doc']}")
        print(f"   EM={row['em']}  F1={row['f1']:.2f}")
        print("-" * 80)


print(">>> CORRECT (EM=1) examples")
show_examples(task2_results_df, n=5, correct=True)

print("\n>>> INCORRECT (EM=0) examples")
show_examples(task2_results_df, n=5, correct=False)

>>> CORRECT (EM=1) examples
Q: What consumer electronics product shares its name with the common name of the fruit Rubus Fructicosis?
   Gold: ['BLACKBERRY', 'Blackberry bush', 'Bush blackberry', 'Bush Blackberry', 'Blackberries', 'Bramble raspberry', 'Rubus subg. Rubus', 'Blackberry (fruit)', 'Black-berry', 'Eubatus', 'Culture of blackberry', 'Rubus subg. Eubatus', 'Blackberry Vines', 'Bramble Raspberry', 'Blackberry', 'blackberry', 'bush blackberry', 'bramble raspberry', 'blackberries', 'culture of blackberry', 'blackberry bush', 'eubatus', 'rubus subg eubatus', 'rubus subg rubus', 'blackberry fruit', 'black berry', 'blackberry vines']
   Pred: 'BlackBerry'
   Top-1 doc: Tartaric acid
   EM=1.0  F1=1.00
--------------------------------------------------------------------------------
Q: What was the name of Shari Lewis' sassy sock puppet?
   Gold: ['Lamb Chop.', 'Lamb Chop', 'Lamb chop', 'Lamb chop (disambiguation)', 'Lamb chops', 'Lambchop', 'lamb chop', 'lambchop', 'lamb chop disamb

## 7. Paper-reproduction comparison

| | Ours (n=300) | Lewis et al. 2020, Table 1 (TriviaQA, RAG-Sequence) |
|---|---|---|
| EM | 0.66 | 56.8 (open-domain) / 68.0 (TQA-Wiki) |
| Retriever | `all-MiniLM-L6-v2`, FAISS flat/cosine, k=5, ~7,660-paragraph from-scratch index | DPR, FAISS HNSW, k∈{5,10}, 21M-chunk Wikipedia |
| Generator | Qwen2.5-7B-Instruct, 4-bit, zero-shot prompted | BART-large (400M), jointly fine-tuned with retriever |

(Discussion of the gap: see report.)

## 8. Summary

EM=0.66, F1=0.698, recall@1=0.51, recall@5=0.67, recall@10=0.73 (n=300). Full discussion and
limitations: see report.